In [ ]:
import pyodbc
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

print(pyodbc.drivers())

driver = next(d for d in ['ODBC Driver 18 for SQL Server',
                          'ODBC Driver 17 for SQL Server',
                          'SQL Server'] if d in pyodbc.drivers())
print(driver)

conn_str = (f"DRIVER={{{driver}}};SERVER=DESKTOP-DUEVVCL\\SQLEXPRESS;"
            "DATABASE=OlistEcommerce;Trusted_Connection=yes;TrustServerCertificate=yes;")
engine = create_engine("mssql+pyodbc:///?odbc_connect=" + quote_plus(conn_str))

with engine.connect() as conn:
    print(conn.execute(text("SELECT DB_NAME()")).scalar())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']
ODBC Driver 17 for SQL Server
OlistEcommerce


In [3]:
import pandas as pd
from pathlib import Path
from sqlalchemy import types as T
from sqlalchemy.dialects import mssql

CLEAN = Path('cleaned')
TABLES = {
    'customers':      'customers_clean.csv',
    'orders':         'orders_clean.csv',
    'order_items':    'order_items_clean.csv',
    'order_payments': 'order_payments_clean.csv',
    'order_reviews':  'order_reviews_clean.csv',
    'products':       'products_clean.csv',
    'sellers':        'sellers_clean.csv',
    'geolocation':    'geolocation_clean.csv',
}
MONEY   = {'price', 'freight_value', 'item_total', 'payment_value'}
ZIPS    = {'customer_zip_code_prefix', 'seller_zip_code_prefix', 'geolocation_zip_code_prefix'}
MAXTEXT = {'review_comment_title', 'review_comment_message'}

def read_table(csv):
    head = pd.read_csv(CLEAN / csv, nrows=0).columns
    str_cols  = [c for c in head if c in ZIPS or (c.endswith('_id') and c != 'order_item_id')]
    date_cols = [c for c in head if c.endswith(('_date', '_timestamp', '_at'))]
    df = pd.read_csv(CLEAN / csv, dtype={c: str for c in str_cols}, parse_dates=date_cols)
    for c in df.columns:                      # whole-number floats with blanks -> Int64
        s = df[c]
        if pd.api.types.is_float_dtype(s) and c not in MONEY and not c.startswith('geolocation_l'):
            if (s.dropna() % 1 == 0).all():
                df[c] = s.astype('Int64')
    return df

def sql_type(name, s):
    if name in ZIPS:    return T.String(5)
    if name in MAXTEXT: return mssql.NVARCHAR(None)
    if name.endswith('_id') and name != 'order_item_id': return T.String(32)
    if pd.api.types.is_bool_dtype(s):             return T.Boolean()
    if pd.api.types.is_datetime64_any_dtype(s):   return mssql.DATETIME2()
    if pd.api.types.is_integer_dtype(s):          return T.Integer()
    if name in MONEY:                             return T.Numeric(12, 2)
    if pd.api.types.is_float_dtype(s):            return T.Float()
    n = s.dropna().astype(str).str.len().max()
    return mssql.NVARCHAR(None) if n > 255 else mssql.NVARCHAR(255)

frames = {t: read_table(f) for t, f in TABLES.items()}
DTYPES = {t: {c: sql_type(c, df[c]) for c in df.columns} for t, df in frames.items()}

for t, df in frames.items():
    print(t, df.shape)
print({c: str(ty.compile(dialect=engine.dialect)) for c, ty in DTYPES['orders'].items()})

customers (99441, 5)
orders (99441, 14)
order_items (112650, 8)
order_payments (103886, 6)
order_reviews (99224, 9)
products (32951, 11)
sellers (3095, 5)
geolocation (19015, 6)
{'order_id': 'VARCHAR(32)', 'customer_id': 'VARCHAR(32)', 'order_status': 'NVARCHAR(255)', 'order_purchase_timestamp': 'DATETIME2', 'order_approved_at': 'DATETIME2', 'order_delivered_carrier_date': 'DATETIME2', 'order_delivered_customer_date': 'DATETIME2', 'order_estimated_delivery_date': 'DATETIME2', 'date_has_issue': 'BIT', 'delivery_time_days': 'INTEGER', 'delivery_delay_days': 'INTEGER', 'is_late': 'NVARCHAR(255)', 'has_sequence_issue': 'BIT', 'has_data_issue': 'BIT'}


In [4]:
engine = create_engine("mssql+pyodbc:///?odbc_connect=" + quote_plus(conn_str),
                       fast_executemany=True)

with engine.begin() as conn:      # drop old relationships so tables can be replaced
    for t, fk in conn.execute(text("SELECT OBJECT_NAME(parent_object_id), name FROM sys.foreign_keys")).fetchall():
        conn.execute(text(f"ALTER TABLE [{t}] DROP CONSTRAINT [{fk}]"))

for t, df in frames.items():
    df.to_sql(t, engine, if_exists='replace', index=False, dtype=DTYPES[t], chunksize=5000)
    print('loaded', t, len(df))

loaded customers 99441
loaded orders 99441
loaded order_items 112650
loaded order_payments 103886
loaded order_reviews 99224
loaded products 32951
loaded sellers 3095
loaded geolocation 19015


In [5]:
PKS = {
    'customers': ['customer_id'], 'orders': ['order_id'],
    'order_items': ['order_id', 'order_item_id'],
    'order_payments': ['order_id', 'payment_sequential'],
    'order_reviews': ['review_id', 'order_id'],
    'products': ['product_id'], 'sellers': ['seller_id'],
    'geolocation': ['geolocation_zip_code_prefix'],
}
FKS = [
    ('orders', 'customer_id', 'customers'), ('order_items', 'order_id', 'orders'),
    ('order_items', 'product_id', 'products'), ('order_items', 'seller_id', 'sellers'),
    ('order_payments', 'order_id', 'orders'), ('order_reviews', 'order_id', 'orders'),
]
with engine.begin() as conn:
    for t, cols in PKS.items():
        for c in cols:
            ddl = DTYPES[t][c].compile(dialect=engine.dialect)
            conn.execute(text(f"ALTER TABLE [{t}] ALTER COLUMN [{c}] {ddl} NOT NULL"))
        conn.execute(text(f"ALTER TABLE [{t}] ADD CONSTRAINT pk_{t} PRIMARY KEY ({', '.join(f'[{c}]' for c in cols)})"))
    for t, c, ref in FKS:
        conn.execute(text(f"ALTER TABLE [{t}] ADD CONSTRAINT fk_{t}_{c} FOREIGN KEY ([{c}]) REFERENCES [{ref}]([{PKS[ref][0]}])"))
print('keys done')

keys done


In [6]:
expected = {'customers': 99441, 'orders': 99441, 'order_items': 112650, 'order_payments': 103886,
            'order_reviews': 99224, 'products': 32951, 'sellers': 3095, 'geolocation': 19015}
with engine.connect() as conn:
    for t, n in expected.items():
        got = conn.execute(text(f"SELECT COUNT(*) FROM [{t}]")).scalar()
        print(f"{t:15} expected {n:>8,}  got {got:>8,}  {'OK' if got == n else 'MISMATCH'}")
    print(conn.execute(text("SELECT SUM(item_total) FROM order_items")).scalar())                # 15843553.24
    print(conn.execute(text("SELECT SUM(payment_value) FROM order_payments")).scalar())          # 16008872.12
    print(conn.execute(text("SELECT COUNT(*) FROM geolocation WHERE geolocation_lat IS NULL")).scalar())  # 4
    print(conn.execute(text("SELECT COUNT(*) FROM customers WHERE LEN(customer_zip_code_prefix) <> 5")).scalar())  # 0

customers       expected   99,441  got   99,441  OK
orders          expected   99,441  got   99,441  OK
order_items     expected  112,650  got  112,650  OK
order_payments  expected  103,886  got  103,886  OK
order_reviews   expected   99,224  got   99,224  OK
products        expected   32,951  got   32,951  OK
sellers         expected    3,095  got    3,095  OK
geolocation     expected   19,015  got   19,015  OK
15843553.24
16008872.12
4
0
